# 01 - Dataset Overview

Basic statistics for the **ClearFairy Cognitive Decision Steps** dataset.

What this notebook covers:
- Step counts per participant and per task type
- Text length distributions for each field
- Coverage of the inferred `rationale` field
- A few example steps

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

# Make scripts/load.py importable when running from notebooks/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from scripts.load import load_dataset

steps = load_dataset()
df = pd.DataFrame(steps)
print(f'{len(df)} steps from {df.participant_id.nunique()} participants')
df.head(2)

## Steps per participant

In [ ]:
per_participant = (
    df.groupby(['participant_id', 'task_type'])
      .size()
      .reset_index(name='n_steps')
      .sort_values('participant_id')
)
per_participant

In [ ]:
ax = per_participant.set_index('participant_id')['n_steps'].plot.bar(
    figsize=(8, 3),
    color=['#4C78A8' if t == 'lab_website' else '#F58518'
           for t in per_participant['task_type']],
    title='Steps per participant (blue = lab_website, orange = shopping_site)',
)
ax.set_ylabel('# steps')
ax.set_xlabel('')
ax.figure.tight_layout()

## Steps by task type

In [ ]:
task_summary = (
    df.groupby('task_type')
      .agg(n_participants=('participant_id', 'nunique'),
           n_steps=('step_index', 'size'),
           steps_per_participant_mean=('participant_id', lambda s: s.value_counts().mean()),
           steps_per_participant_min=('participant_id', lambda s: s.value_counts().min()),
           steps_per_participant_max=('participant_id', lambda s: s.value_counts().max()))
      .round(1)
)
task_summary

## Text length distributions

How many characters does each field hold?

In [ ]:
for col in ['decision_and_actions', 'rationale', 'progression']:
    df[f'{col}_len'] = df[col].str.len()

df[['decision_and_actions_len', 'rationale_len', 'progression_len']].describe().round(1)

In [ ]:
ax = df[['decision_and_actions_len', 'rationale_len', 'progression_len']].plot.box(
    figsize=(8, 3),
    title='Character length per field',
)
ax.set_ylabel('characters')
ax.figure.tight_layout()

## Rationale coverage

The `rationale` field is inferred and not always produced. How often is it present?

In [ ]:
df['has_rationale'] = df['rationale'].str.strip().astype(bool)
coverage = (
    df.groupby('participant_id')['has_rationale']
      .mean()
      .mul(100)
      .round(1)
      .reset_index(name='rationale_coverage_%')
)
overall = df['has_rationale'].mean() * 100
print(f'Overall rationale coverage: {overall:.1f}%')
coverage

## Example steps

One example from a `lab_website` participant and one from a `shopping_site` participant.

In [ ]:
for task in ['lab_website', 'shopping_site']:
    sample = df[(df.task_type == task) & (df.has_rationale)].iloc[0]
    print(f"=== {task} | {sample.participant_id} | step {sample.step_index} ===")
    print(f"decision_and_actions: {sample.decision_and_actions}")
    print(f"rationale:            {sample.rationale}")
    print(f"progression:          {sample.progression}")
    print()